## Simulate SNIa avec GridSurvey

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS
- **Creation date:** 2026-07-30
- **mac**: python kernel = conda_py313

==> Not GridSurvey ne marche pas

## Import

In [ ]:
import skysurvey
from shapely import geometry
from astropy.time import Time
import pandas as pd
import numpy as np

## Configuration

In [ ]:
TSTART = "2026-09-01"
TSTOP = "2027-09-30"

In [ ]:
tstart_mjd = Time(TSTART).mjd
tstop_mjd = Time(TSTOP).mjd

In [ ]:
print(tstart_mjd, tstop_mjd)

In [ ]:
# footprint

footprint = geometry.Point(0, 0).buffer(2)

# fields

radec = {
    "C1": {"dec": -27.11161, "ra": 54.274292 + 180},
    "C2": {"dec": -29.08839, "ra": 54.274292 + 180},
    "C3": {"dec": -28.10000, "ra": 52.648417 + 180},
}

field_map = {"C1": 0, "C2": 1, "C3": 2}
radec_int = {
    0: {"dec": -27.11161, "ra": 54.274292 + 180},
    1: {"dec": -29.08839, "ra": 54.274292 + 180},
    2: {"dec": -28.10000, "ra": 52.648417 + 180},
}

# observing logs
size = 10_000

In [ ]:
df_data = pd.DataFrame(
    {
        "gain": np.ones(size),
        "zp": np.full(size, 30),
        "skynoise": np.random.normal(size=size, loc=200, scale=20),
        "mjd": np.sort(np.random.uniform(tstart_mjd, tstop_mjd, size=size)),
        "band": np.random.choice(["lsstu", "lsstg", "lsstr", "lssti", "lsstz", "lssty"], size=size),
        "fieldid": np.random.choice(list(field_map.values()), size=size),
    }
)
df_data["fieldid"] = df_data["fieldid"].astype("int64")

## Define the target

In [ ]:
def assign_fieldid(ra, dec, radec_dict):
    # assignation naïve au champ le plus proche
    coords = np.array([(v["ra"], v["dec"]) for v in radec_dict.values()])
    ids = list(radec_dict.keys())

    d2 = (coords[:, 0] - ra) ** 2 + (coords[:, 1] - dec) ** 2
    return ids[np.argmin(d2)]

In [ ]:
snia = skysurvey.SNeIa()
data = snia.draw(size=10_000, inplace=True)  # inplace sets snia.data
data.head(5)  # see also snia.data

In [ ]:
snii = skysurvey.SNeII()
data = snii.draw(size=10_000, inplace=True)  # inplace sets snia.data
data.head(5)  # see also self.data

In [ ]:
for t in [snia, snii]:
    t.data["fieldid"] = [assign_fieldid(ra, dec, radec_int) for ra, dec in zip(t.data["ra"], t.data["dec"])]
    t.data["fieldid"] = t.data["fieldid"].astype("int64")

In [ ]:
snia.data

In [ ]:
# for t in [snia, snii]:
#    t.data = t.data.drop(columns=["fieldid"])

In [ ]:
targets = skysurvey.TargetCollection([snia, snii])

print(targets.data.columns)

## Define the survey

In [ ]:
# Load a GridSurvey
survey = skysurvey.GridSurvey.from_pointings(df_data, radec, footprint=footprint)

## Dataset

In [ ]:
def fixed_from_targets_and_survey(targets, survey):
    targets = skysurvey.TargetCollection(targets)

    # --- récupération du mapping interne ---
    dfieldids_ = survey.fieldids.get_fieldids(targets.data)

    # 🔥 FIX ICI
    dfieldids_ = dfieldids_.drop(columns=["fieldid"], errors="ignore")

    targets_data = targets.data.merge(dfieldids_, left_index=True, right_index=True)

    target_fields = np.stack(targets_data[survey.fieldids.names].values, dtype="int")

    return targets_data, target_fields

In [ ]:
# targets_data, target_fields = fixed_from_targets_and_survey([snia, snii], survey)

In [ ]:
# simply pass a list of targets
dset = skysurvey.DataSet.from_targets_and_survey(targets, survey)
dset.data